# StatsAmerica BEA and CEW Raw Data Quality

**Purpose.** Audit the files ingested for this provider and the corresponding `raw.*`
DuckDB tables before any normalization, blending, or analytical transformation.

This notebook covers the supplied files/tables, observation grain, date and geography
coverage, column types and meanings, missingness and suppression, duplicate/invalid
keys, numeric ranges, suspicious values, source limitations, and downstream readiness.

## Setup and provider rules

In [1]:
from pathlib import Path
import re
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

ROOT = Path.cwd()
while not (ROOT / "data" / "quoll.duckdb").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DB_PATH = ROOT / "data" / "quoll.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)

PROVIDER = 'statsamerica'
TABLE_PATTERNS = ['statsamerica_%']
PRIMARY_PATTERNS = ['statsamerica_bea_per_capita_income', 'statsamerica_bea_personal_income', 'statsamerica_cew_total_ownership', 'statsamerica_population_components']
KEY_CANDIDATES = [['Statefips', 'Countyfips', 'Year', 'Linecode'], ['Statefips', 'Countyfips', 'Year', 'NAICS Code', 'Ownership Code'], ['Statefips', 'Countyfips', 'Year']]
DATE_CANDIDATES = ['Year']
GEO_CANDIDATES = ['Statefips', 'Countyfips', 'Description']
NUMERIC_HINTS = ['Year', 'Data', 'BEA Per Capita Personal Income', 'Employment', 'Wages', 'Average Wage', 'Births', 'Deaths']
SUPPRESSION_CODES = ['', '(D)', '(L)', '(N)', 'null']

def matches(name, patterns):
    return any(re.fullmatch(pattern.replace("%", ".*"), name, flags=re.I) for pattern in patterns)

def qi(value):
    return '"' + value.replace('"', '""') + '"'

raw_tables = con.execute(
    "SELECT table_name FROM information_schema.tables "
    "WHERE table_schema = 'raw' ORDER BY table_name"
).df()["table_name"].tolist()
provider_tables = [name for name in raw_tables if matches(name, TABLE_PATTERNS)]
primary_tables = [name for name in provider_tables if matches(name, PRIMARY_PATTERNS)]
provider_tables, primary_tables

(['statsamerica_bea_per_capita_income',
  'statsamerica_bea_personal_income',
  'statsamerica_cew_total_ownership',
  'statsamerica_population_components'],
 ['statsamerica_bea_per_capita_income',
  'statsamerica_bea_personal_income',
  'statsamerica_cew_total_ownership',
  'statsamerica_population_components'])

## Files and tables supplied

In [2]:
file_inventory = con.execute(
    '''
    SELECT table_name, filename, source_folder, source_path,
           loaded_at, row_count, detected_columns,
           upstream_source_url, content_sha256
    FROM meta.files
    WHERE table_schema = 'raw'
    ORDER BY table_name
    '''
).df()
file_inventory = file_inventory.loc[file_inventory["table_name"].isin(provider_tables)]

table_rows = []
for table in provider_tables:
    row_count = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    column_count = con.execute(
        "SELECT count(*) FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).fetchone()[0]
    table_rows.append({"table_name": table, "rows": row_count, "columns": column_count,
                       "primary_data_table": table in primary_tables})
table_inventory = pd.DataFrame(table_rows)
display(file_inventory)
display(table_inventory)

,table_name,filename,source_folder,source_path,loaded_at,row_count,detected_columns,upstream_source_url,content_sha256
56,statsamerica_bea_per_capita_income,"BEA - US, States, Counties - Per Capita Income...",statsamerica,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:55:44.560588+00:00,72910,"[""IBRC_Geo_ID"", ""Statefips"", ""Countyfips"", ""De...",None,1ecd549315fddac217b613cfea2c06a38716bed856f5ec...
57,statsamerica_bea_personal_income,"BEA - US, States, Counties - Personal Income.csv",statsamerica,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:55:53.184068+00:00,9551210,"[""IBRC_GEO_ID"", ""Statefips"", ""Countyfips"", ""De...",None,80e1c3d7a61ec37ab741b5030237a352d558cc84d53244...
58,statsamerica_cew_total_ownership,"CEW - US, States, Counties - Total Ownership.csv",statsamerica,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:57:29.070355+00:00,58515662,"[""IBRC_GEO_ID"", ""Statefips"", ""Countyfips"", ""De...",None,6d3323a2b29b57c2480ce829078f985aafa9a8b2e4f087...
59,statsamerica_population_components,"Components of Population Change - U.S., States...",statsamerica,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:55:44.439023+00:00,122267,"[""IBRC_Geo_ID"", ""Statefips"", ""Countyfips"", ""De...",None,33d62669da2b7e7919b954899e4bafb122c46dde6a5cb5...


,table_name,rows,columns,primary_data_table
0,statsamerica_bea_per_capita_income,72910,6,True
1,statsamerica_bea_personal_income,9551210,9,True
2,statsamerica_cew_total_ownership,58515662,16,True
3,statsamerica_population_components,122267,10,True


## Observation grain

County-year-linecode for BEA personal income, county-year-industry-ownership for CEW, and county-year for per-capita income and population components.

The checks below infer candidate keys from the raw columns. A repeated candidate key is
reported rather than silently removed because some provider tables legitimately contain
additional dimensions.

## Column types and meanings

In [3]:
schema_frames = []
for table in primary_tables:
    schema = con.execute(f"DESCRIBE raw.{qi(table)}").df()
    schema.insert(0, "table_name", table)
    schema["inferred_meaning"] = (
        schema["column_name"].str.replace("_", " ", regex=False)
        .str.replace(r"(?<=[a-z])(?=[A-Z])", " ", regex=True)
        .str.strip()
    )
    schema_frames.append(schema)
schema_inventory = pd.concat(schema_frames, ignore_index=True) if schema_frames else pd.DataFrame()
display(schema_inventory)

,table_name,column_name,column_type,null,key,default,extra,inferred_meaning
0,statsamerica_bea_per_capita_income,IBRC_Geo_ID,VARCHAR,YES,None,None,None,IBRC Geo ID
1,statsamerica_bea_per_capita_income,Statefips,VARCHAR,YES,None,None,None,Statefips
2,statsamerica_bea_per_capita_income,Countyfips,VARCHAR,YES,None,None,None,Countyfips
3,statsamerica_bea_per_capita_income,Description,VARCHAR,YES,None,None,None,Description
4,statsamerica_bea_per_capita_income,Year,VARCHAR,YES,None,None,None,Year
5,statsamerica_bea_per_capita_income,BEA Per Capita Personal Income,VARCHAR,YES,None,None,None,BEA Per Capita Personal Income
6,statsamerica_bea_personal_income,IBRC_GEO_ID,VARCHAR,YES,None,None,None,IBRC GEO ID
7,statsamerica_bea_personal_income,Statefips,VARCHAR,YES,None,None,None,Statefips
8,statsamerica_bea_personal_income,Countyfips,VARCHAR,YES,None,None,None,Countyfips
9,statsamerica_bea_personal_income,Description,VARCHAR,YES,None,None,None,Description


## Date and geographic coverage

In [4]:
coverage_rows = []
for table in primary_tables:
    columns = con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"].tolist()
    row = {"table_name": table}
    for column in DATE_CANDIDATES:
        if column in columns:
            normalized_column = column.lower()
            if normalized_column == "year" or normalized_column.endswith("_year"):
                coverage_type = "INTEGER"
            elif normalized_column == "month" or normalized_column.endswith("_month"):
                coverage_type = "INTEGER"
            else:
                coverage_type = "TIMESTAMP"
            values = con.execute(
                f"SELECT min(try_cast({qi(column)} AS {coverage_type})), "
                f"max(try_cast({qi(column)} AS {coverage_type})) "
                f"FROM raw.{qi(table)}"
            ).fetchone()
            row[f"{column}_min"] = values[0]
            row[f"{column}_max"] = values[1]
    for column in GEO_CANDIDATES:
        if column in columns:
            row[f"{column}_distinct"] = con.execute(
                f"SELECT count(DISTINCT {qi(column)}) FROM raw.{qi(table)}"
            ).fetchone()[0]
    coverage_rows.append(row)
coverage = pd.DataFrame(coverage_rows)
display(coverage)

,table_name,Year_min,Year_max,Statefips_distinct,Countyfips_distinct,Description_distinct
0,statsamerica_bea_per_capita_income,2001,2023,52,301,3169
1,statsamerica_bea_personal_income,2001,2023,44,301,3169
2,statsamerica_cew_total_ownership,2001,2024,43,334,3211
3,statsamerica_population_components,1981,2025,52,334,3210


## Missingness and suppression codes

In [5]:
missing_rows = []
suppression_rows = []
suppression_sql = ", ".join("?" for _ in SUPPRESSION_CODES)
for table in primary_tables:
    columns = con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=? ORDER BY ordinal_position", [table]
    ).df()["column_name"].tolist()
    row_count = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    # Profile all columns for compact tables and the first 80 for unusually wide sources.
    for column in columns[:80]:
        null_count, blank_count = con.execute(
            f"SELECT count(*) FILTER (WHERE {qi(column)} IS NULL), "
            f"count(*) FILTER (WHERE trim(cast({qi(column)} AS VARCHAR))='') "
            f"FROM raw.{qi(table)}"
        ).fetchone()
        missing_rows.append({
            "table_name": table, "column_name": column,
            "missing_count": null_count + blank_count,
            "missing_pct": (null_count + blank_count) / row_count * 100 if row_count else np.nan,
        })
        if SUPPRESSION_CODES:
            suppressed = con.execute(
                f"SELECT count(*) FROM raw.{qi(table)} "
                f"WHERE trim(cast({qi(column)} AS VARCHAR)) IN ({suppression_sql})",
                SUPPRESSION_CODES,
            ).fetchone()[0]
            if suppressed:
                suppression_rows.append({
                    "table_name": table, "column_name": column,
                    "suppression_or_sentinel_count": suppressed,
                })
missingness = pd.DataFrame(missing_rows).sort_values(
    ["missing_pct", "table_name"], ascending=[False, True]
)
suppression = pd.DataFrame(suppression_rows)
display(missingness)
display(suppression if not suppression.empty else pd.DataFrame(
    {"result": ["No configured literal suppression codes were present in profiled columns; nulls remain material."]}
))

,table_name,column_name,missing_count,missing_pct
39,statsamerica_population_components,Net Domestic Migration,59885,48.978874
38,statsamerica_population_components,Net International Migration,14246,11.651549
28,statsamerica_cew_total_ownership,Average Wage,805222,1.376079
29,statsamerica_cew_total_ownership,Average Weekly Wage,805222,1.376079
0,statsamerica_bea_per_capita_income,IBRC_Geo_ID,0,0.000000
1,statsamerica_bea_per_capita_income,Statefips,0,0.000000
2,statsamerica_bea_per_capita_income,Countyfips,0,0.000000
3,statsamerica_bea_per_capita_income,Description,0,0.000000
4,statsamerica_bea_per_capita_income,Year,0,0.000000
5,statsamerica_bea_per_capita_income,BEA Per Capita Personal Income,0,0.000000


,result
0,No configured literal suppression codes were p...


## Duplicate or invalid keys

In [6]:
key_rows = []
for table in primary_tables:
    columns = set(con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"])
    keys = next((candidate for candidate in KEY_CANDIDATES if set(candidate).issubset(columns)), [])
    if not keys:
        key_rows.append({"table_name": table, "candidate_key": None,
                         "duplicate_key_groups": np.nan, "invalid_key_rows": np.nan})
        continue
    key_expr = ", ".join(qi(column) for column in keys)
    invalid = " OR ".join(
        f"{qi(column)} IS NULL OR trim(cast({qi(column)} AS VARCHAR))=''" for column in keys
    )
    duplicate_groups = con.execute(
        f"SELECT count(*) FROM (SELECT {key_expr}, count(*) n "
        f"FROM raw.{qi(table)} GROUP BY {key_expr} HAVING count(*) > 1)"
    ).fetchone()[0]
    invalid_rows = con.execute(
        f"SELECT count(*) FROM raw.{qi(table)} WHERE {invalid}"
    ).fetchone()[0]
    key_rows.append({"table_name": table, "candidate_key": " + ".join(keys),
                     "duplicate_key_groups": duplicate_groups,
                     "invalid_key_rows": invalid_rows})
key_quality = pd.DataFrame(key_rows)
display(key_quality)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,table_name,candidate_key,duplicate_key_groups,invalid_key_rows
0,statsamerica_bea_per_capita_income,Statefips + Countyfips + Year,23,0
1,statsamerica_bea_personal_income,Statefips + Countyfips + Year + Linecode,2163334,0
2,statsamerica_cew_total_ownership,Statefips + Countyfips + Year + NAICS Code + O...,13122641,0
3,statsamerica_population_components,Statefips + Countyfips + Year,6434,0


## Numeric ranges and suspicious values

In [7]:
numeric_rows = []
for table in primary_tables:
    columns = set(con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"])
    for column in [name for name in NUMERIC_HINTS if name in columns]:
        numeric = (
            f"try_cast(replace(trim(cast({qi(column)} AS VARCHAR)), ',', '') AS DOUBLE)"
        )
        result = con.execute(
            f"SELECT count(*) FILTER (WHERE {numeric} IS NOT NULL), "
            f"min({numeric}), max({numeric}), "
            f"count(*) FILTER (WHERE {numeric} < 0) "
            f"FROM raw.{qi(table)}"
        ).fetchone()
        numeric_rows.append({
            "table_name": table, "column_name": column,
            "numeric_count": result[0], "minimum": result[1],
            "maximum": result[2], "negative_count": result[3],
            "review_flag": (
                "review negative values/sentinels" if result[3] else
                "review extreme min/max against provider definition"
            ),
        })
numeric_ranges = pd.DataFrame(numeric_rows)
display(numeric_ranges)

,table_name,column_name,numeric_count,minimum,maximum,negative_count,review_flag
0,statsamerica_bea_per_capita_income,Year,72910,2001.0,2.023000e+03,0,review extreme min/max against provider defini...
1,statsamerica_bea_per_capita_income,BEA Per Capita Personal Income,72910,0.0,4.717510e+05,0,review extreme min/max against provider defini...
2,statsamerica_bea_personal_income,Year,9551210,2001.0,2.023000e+03,0,review extreme min/max against provider defini...
3,statsamerica_bea_personal_income,Data,9551210,-263181243.0,2.338027e+10,347392,review negative values/sentinels
4,statsamerica_cew_total_ownership,Year,58515662,2001.0,2.024000e+03,0,review extreme min/max against provider defini...
5,statsamerica_cew_total_ownership,Employment,58515662,0.0,1.548703e+08,0,review extreme min/max against provider defini...
6,statsamerica_cew_total_ownership,Wages,58515662,0.0,1.170889e+13,0,review extreme min/max against provider defini...
7,statsamerica_cew_total_ownership,Average Wage,57710440,0.0,1.260048e+07,0,review extreme min/max against provider defini...
8,statsamerica_population_components,Year,122267,1981.0,2.025000e+03,0,review extreme min/max against provider defini...
9,statsamerica_population_components,Births,122267,0.0,3.682013e+07,0,review extreme min/max against provider defini...


## Source-specific limitations

BEA and CEW concepts have different universes and revision schedules, CEW values can be disclosure-suppressed, and exact-year joins reduce coverage when releases are not synchronized.

## Downstream readiness

**Assessment: PASS WITH LIMITATIONS after filtering county rows, honoring disclosure flags, selecting documented BEA linecodes/CEW industries, and retaining exact source years.**

This assessment is conditional on the displayed inventories and checks. The normalized
`mart.*` builders—not this notebook—own parsing, suppression handling, geographic
resolution, deduplication, and downstream transformations.

In [8]:
con.close()